# nyc311 SDK Quickstart

This notebook starts with the simplest possible workflow, then layers on more advanced analysis.

Setup for local use:
- run `uv sync --extra science --group notebooks`
- select the project `.venv` / `nyc311` kernel in Jupyter or VS Code

Levels in this notebook:
1. fetch a small live sample and inspect it as a DataFrame
2. extract and aggregate complaint topics
3. inspect resolution gaps
4. audit topic coverage so `other` is visible
5. add custom rules for a new complaint type
6. score anomalies and render a notebook-native report summary

You can stop after any section and still have a useful, working workflow.

In [9]:
from datetime import date

from IPython.display import Markdown, display

import nyc311

records = nyc311.fetch_service_requests(
    filters=nyc311.ServiceRequestFilter(
        start_date=date(2025, 1, 1),
        end_date=date(2025, 1, 31),
        geography=nyc311.GeographyFilter("borough", nyc311.BOROUGH_BROOKLYN),
        complaint_types=("Noise - Residential",),
    ),
    socrata_config=nyc311.SocrataConfig(page_size=100, max_pages=1),
)

records_df = nyc311.records_to_dataframe(records)

In [10]:
print(f"Found {len(records)} records")

records_df.head()

Found 100 records


,service_request_id,created_date,complaint_type,descriptor,borough,community_district,resolution_description
0,63584096,2025-01-01,Noise - Residential,Loud Music/Party,BROOKLYN,01 BROOKLYN,The Police Department responded to the complai...
1,63572426,2025-01-01,Noise - Residential,Banging/Pounding,BROOKLYN,16 BROOKLYN,The Police Department responded to the complai...
2,63575947,2025-01-01,Noise - Residential,Loud Music/Party,BROOKLYN,18 BROOKLYN,The Police Department responded to the complai...
3,63572417,2025-01-01,Noise - Residential,Banging/Pounding,BROOKLYN,01 BROOKLYN,The Police Department responded to the complai...
4,63580908,2025-01-01,Noise - Residential,Loud Music/Party,BROOKLYN,03 BROOKLYN,The Police Department responded to the complai...


In [2]:
records_df[
    [
        "service_request_id",
        "created_date",
        "complaint_type",
        "descriptor",
        "borough",
        "community_district",
    ]
].head(10)

,service_request_id,created_date,complaint_type,descriptor,borough,community_district
0,63584096,2025-01-01,Noise - Residential,Loud Music/Party,BROOKLYN,01 BROOKLYN
1,63572426,2025-01-01,Noise - Residential,Banging/Pounding,BROOKLYN,16 BROOKLYN
2,63575947,2025-01-01,Noise - Residential,Loud Music/Party,BROOKLYN,18 BROOKLYN
3,63572417,2025-01-01,Noise - Residential,Banging/Pounding,BROOKLYN,01 BROOKLYN
4,63580908,2025-01-01,Noise - Residential,Loud Music/Party,BROOKLYN,03 BROOKLYN
5,63582776,2025-01-01,Noise - Residential,Loud Music/Party,BROOKLYN,15 BROOKLYN
6,63582496,2025-01-01,Noise - Residential,Loud Music/Party,BROOKLYN,03 BROOKLYN
7,63584323,2025-01-01,Noise - Residential,Loud Music/Party,BROOKLYN,03 BROOKLYN
8,63574056,2025-01-01,Noise - Residential,Banging/Pounding,BROOKLYN,06 BROOKLYN
9,63572630,2025-01-01,Noise - Residential,Loud Music/Party,BROOKLYN,16 BROOKLYN


In [3]:
assignments = nyc311.extract_topics(
    records,
    nyc311.TopicQuery("Noise - Residential", top_n=10),
)
summaries = nyc311.aggregate_by_geography(
    assignments,
    geography="community_district",
)
summaries_df = nyc311.summaries_to_dataframe(summaries)

dominant_topics_df = summaries_df[summaries_df["is_dominant_topic"]].sort_values(
    ["complaint_count", "share_of_geography"],
    ascending=[False, False],
)
display(dominant_topics_df.head(10))

plot_df = dominant_topics_df.head(10).sort_values("complaint_count")
try:
    import matplotlib.pyplot as plt

    ax = plot_df.plot.barh(
        x="geography_value",
        y="complaint_count",
        figsize=(9, 4),
        color="#4C78A8",
        legend=False,
        title="Dominant noise topic counts by community district",
    )
    ax.set_xlabel("Complaint count")
    ax.set_ylabel("Community district")
    plt.show()
except ImportError:
    display(plot_df[["geography_value", "topic", "complaint_count"]])

,geography,geography_value,complaint_type,topic,complaint_count,geography_total_count,share_of_geography,topic_rank,is_dominant_topic
0,community_district,01 BROOKLYN,Noise - Residential,party_music,10,13,0.769231,1,True
2,community_district,02 BROOKLYN,Noise - Residential,banging,2,3,0.666667,1,True
4,community_district,03 BROOKLYN,Noise - Residential,party_music,8,8,1.000000,1,True
5,community_district,04 BROOKLYN,Noise - Residential,party_music,7,7,1.000000,1,True
6,community_district,05 BROOKLYN,Noise - Residential,party_music,9,9,1.000000,1,True
7,community_district,06 BROOKLYN,Noise - Residential,party_music,2,3,0.666667,1,True
9,community_district,07 BROOKLYN,Noise - Residential,party_music,3,5,0.600000,1,True
11,community_district,08 BROOKLYN,Noise - Residential,party_music,4,6,0.666667,1,True
13,community_district,09 BROOKLYN,Noise - Residential,banging,3,5,0.600000,1,True
15,community_district,10 BROOKLYN,Noise - Residential,party_music,4,5,0.800000,1,True


In [4]:
resolution_summaries = nyc311.analyze_resolution_gaps(
    records,
    [record for record in records if record.resolution_description is not None],
)
resolution_df = nyc311.gaps_to_dataframe(resolution_summaries)
display(resolution_df.head(10))

try:
    import matplotlib.pyplot as plt

    ax = resolution_df.head(10).plot.bar(
        x="complaint_type",
        y="resolution_rate",
        figsize=(8, 4),
        color="#59A14F",
        legend=False,
        title="Resolution rate for the current sample",
    )
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Resolution rate")
    ax.set_xlabel("Complaint type")
    plt.xticks(rotation=45, ha="right")
    plt.show()
except ImportError:
    display(resolution_df[["complaint_type", "resolution_rate"]].head(10))

,geography,geography_value,complaint_type,total_request_count,resolved_request_count,unresolved_request_count,unresolved_share,resolution_rate
0,borough,BROOKLYN,Noise - Residential,100,100,0,0.0,1.0


In [5]:
coverage = nyc311.analyze_topic_coverage(
    records,
    nyc311.TopicQuery("Noise - Residential", top_n=10),
)
coverage_df = nyc311.coverage_to_dataframe([coverage])
display(coverage_df)

display(
    Markdown(
        "\n".join(
            [
                "### Coverage Audit",
                f"- Matched records: {coverage.matched_records}/{coverage.total_records}",
                f"- Coverage rate: {coverage.coverage_rate:.1%}",
                f"- Top unmatched descriptors: {list(coverage.top_unmatched_descriptors) or 'None'}",
            ]
        )
    )
)

,complaint_type,total_records,matched_records,other_records,coverage_rate,top_unmatched_descriptors
0,Noise - Residential,100,97,3,0.97,"[(Loud Talking, 3)]"


In [6]:
custom_rules = (
    ("hydrant_issue", ("hydrant", "low water pressure")),
    ("leak", ("leak", "leaking")),
)

water_records = [
    nyc311.ServiceRequestRecord(
        service_request_id="demo-1",
        created_date=date(2025, 1, 1),
        complaint_type="Water System",
        descriptor="Low water pressure near hydrant",
        borough=nyc311.BOROUGH_BROOKLYN,
        community_district="01 BROOKLYN",
    ),
    nyc311.ServiceRequestRecord(
        service_request_id="demo-2",
        created_date=date(2025, 1, 2),
        complaint_type="Water System",
        descriptor="Leaking hydrant on corner",
        borough=nyc311.BOROUGH_BROOKLYN,
        community_district="01 BROOKLYN",
    ),
    nyc311.ServiceRequestRecord(
        service_request_id="demo-3",
        created_date=date(2025, 1, 3),
        complaint_type="Water System",
        descriptor="Pressure issue in building basement",
        borough=nyc311.BOROUGH_BROOKLYN,
        community_district="01 BROOKLYN",
    ),
]

before_custom_rules = nyc311.analyze_topic_coverage(
    water_records,
    nyc311.TopicQuery("Water System", top_n=10),
)
after_custom_rules = nyc311.analyze_topic_coverage(
    water_records,
    nyc311.TopicQuery("Water System", top_n=10),
    custom_rules=custom_rules,
)
nyc311.coverage_to_dataframe([before_custom_rules, after_custom_rules])

,complaint_type,total_records,matched_records,other_records,coverage_rate,top_unmatched_descriptors
0,Water System,3,3,0,1.000000,[]
1,Water System,3,2,1,0.666667,"[(Pressure issue in building basement, 1)]"


In [7]:
anomalies = nyc311.detect_anomalies(
    summaries,
    nyc311.AnalysisWindow(days=30),
    z_threshold=1.5,
)
anomalies_df = nyc311.anomalies_to_dataframe(anomalies)

display(
    Markdown(
        "\n".join(
            [
                "## Sample Report Card",
                f"- Sample size: {len(records)} records",
                f"- Dominant districts shown above: {len(dominant_topics_df)}",
                f"- Topic coverage: {coverage.coverage_rate:.1%}",
                f"- Anomalies above threshold: {int(anomalies_df['is_anomaly'].sum())}",
            ]
        )
    )
)

display(anomalies_df.head(10))

plot_df = anomalies_df.head(10).sort_values("z_score")
try:
    import matplotlib.pyplot as plt

    ax = plot_df.plot.barh(
        x="geography_value",
        y="z_score",
        figsize=(9, 4),
        color="#E15759",
        legend=False,
        title="Top topic anomaly scores in the sample",
    )
    ax.set_xlabel("z-score")
    ax.set_ylabel("Community district")
    plt.show()
except ImportError:
    display(plot_df[["geography_value", "topic", "z_score", "is_anomaly"]])

,geography,geography_value,complaint_type,topic,complaint_count,geography_total_count,share_of_geography,topic_rank,z_score,is_anomaly,window_days,anomaly_threshold
0,community_district,01 BROOKLYN,Noise - Residential,party_music,10,13,0.769231,1,2.781470,True,30,1.5
1,community_district,05 BROOKLYN,Noise - Residential,party_music,9,9,1.000000,1,2.376892,True,30,1.5
2,community_district,03 BROOKLYN,Noise - Residential,party_music,8,8,1.000000,1,1.972315,True,30,1.5
3,community_district,04 BROOKLYN,Noise - Residential,party_music,7,7,1.000000,1,1.567737,True,30,1.5
4,community_district,18 BROOKLYN,Noise - Residential,party_music,6,6,1.000000,1,1.163160,False,30,1.5
5,community_district,02 BROOKLYN,Noise - Residential,party_music,1,3,0.333333,2,-0.859727,False,30,1.5
6,community_district,06 BROOKLYN,Noise - Residential,banging,1,3,0.333333,2,-0.859727,False,30,1.5
7,community_district,10 BROOKLYN,Noise - Residential,other,1,5,0.200000,2,-0.859727,False,30,1.5
8,community_district,11 BROOKLYN,Noise - Residential,banging,1,4,0.250000,2,-0.859727,False,30,1.5
9,community_district,11 BROOKLYN,Noise - Residential,other,1,4,0.250000,3,-0.859727,False,30,1.5


In [8]:
# Optional: persist data or summaries later.
# The notebook intentionally renders tables and markdown inline first.
#
# nyc311.export_service_requests_csv(
#     records,
#     nyc311.ExportTarget("csv", "brooklyn-noise-snapshot.csv"),
# )
#
# nyc311.export_topic_table(
#     summaries,
#     nyc311.ExportTarget("csv", "brooklyn-noise-topics.csv"),
# )
#
# nyc311.export_anomalies(
#     anomalies,
#     nyc311.ExportTarget("csv", "brooklyn-noise-anomalies.csv"),
# )